In [ ]:
# System setup and package verification
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add the project root to path (going up 2 levels from simplicial tutorial folder)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..'))


if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added project root to path: {project_root}")

# Verify the import works
try:
    import toposheafx
    print(f"Successfully imported toposheafx from: {toposheafx.__file__}")
except ImportError as e:
    print(f"Failed to import toposheafx: {e}")

## Imports

In [ ]:

"""
Simple Copresheaf Neural Network Example
==========================================
A straightforward example demonstrating copresheaf message passing on a simplicial complex
using TopoSheafX and TopoNetX with custom linear maps for vertex feature learning.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from toponetx.classes import SimplicialComplex

# Import TopoSheafX components
from toposheafx.nn.layer import CMPNNLayer
from toposheafx.nn.model import NodeCMPNN

## Create Simplicial Complex for Example Simplicial Copresheaf Neural Network

In [ ]:



def create_simple_simplicial_complex():
    """
    Create a simple triangle simplicial complex using TopoNetX.
    
    Structure:
    - 4 vertices (0, 1, 2, 3)
    - Edges connecting vertices
    - 1 triangle (0, 1, 2)
    
    Returns:
        SimplicialComplex: A simple simplicial complex
    """
    # Define simplices: vertices, edges, and triangles
    simplices = [
        [0], [1], [2], [3],           # 0-simplices (vertices)
        [0, 1], [1, 2], [0, 2], [2, 3],  # 1-simplices (edges)
        [0, 1, 2]                      # 2-simplices (triangle)
    ]
    
    # Create simplicial complex
    sc = SimplicialComplex(simplices)
    
    print(f"Created simplicial complex with:")
    print(f"  - {len(list(sc.skeleton(0)))} vertices")
    print(f"  - {len(list(sc.skeleton(1)))} edges")
    print(f"  - {len(list(sc.skeleton(2)))} triangles")
    
    return sc




## Format Adjacency Information

In [ ]:

def simplicial_complex_to_edge_index(sc):
    """
    Convert simplicial complex to PyTorch Geometric edge_index format.
    
    Args:
        sc: SimplicialComplex from TopoNetX
        
    Returns:
        torch.Tensor: Edge indices [2, num_edges]
    """
    edges = []
    for edge in sc.skeleton(1):
        edge_list = list(edge)
        if len(edge_list) == 2:
            # Add both directions for undirected edges
            edges.append([edge_list[0], edge_list[1]])
            edges.append([edge_list[1], edge_list[0]])
    
    if not edges:
        return torch.empty((2, 0), dtype=torch.long)
    
    edge_index = torch.tensor(edges, dtype=torch.long).t()
    return edge_index




## Create Copresheaf Message Passing NeuralNetwork

In [ ]:

class SimpleCopresheafNN(nn.Module):
    """
    Simple Copresheaf Neural Network with custom linear maps.
    
    This model uses CMPNNLayer from TopoSheafX which implements:
    h_x^{l+1} = β(h_x^l, ⊕_{(y→x)∈E} α(h_x^l, ρ_{y→x}(h_y^l)))
    
    Where:
    - ρ_{y→x}: Copresheaf morphism (linear transport map)
    - α: Message function
    - β: Update function
    - ⊕: Aggregation (sum, mean, etc.)
    """
    
    def __init__(self, in_dim, hidden_dim, out_dim, num_layers=2):
        super().__init__()
        
        # Input projection
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        
        # Copresheaf layers from TopoSheafX
        self.layers = nn.ModuleList([
            CMPNNLayer(
                in_dim=hidden_dim,
                out_dim=hidden_dim,
                num_neighborhoods=1  # Single neighborhood (edges)
            )
            for _ in range(num_layers)
        ])
        
        # Output projection
        self.output_proj = nn.Linear(hidden_dim, out_dim)
    
    def forward(self, x, edge_index_list):
        """
        Forward pass through copresheaf layers.
        
        Args:
            x: Node features [num_nodes, in_dim]
            edge_index_list: List of edge indices
            
        Returns:
            Node predictions [num_nodes, out_dim]
        """
        # Project input to hidden dimension
        x = F.relu(self.input_proj(x))
        
        # Apply copresheaf message passing layers
        for layer in self.layers:
            x_new = layer(x, edge_index_list)
            x = x + x_new  # Residual connection
            x = F.relu(x)
        
        # Project to output
        out = self.output_proj(x)
        
        return out


## Train and Test CMPNN

In [ ]:

print("=" * 60)
print("Simple Copresheaf Neural Network Example")
print("=" * 60)
print()

# 1. Create simplicial complex using TopoNetX
print("Step 1: Creating simplicial complex...")
sc = create_simple_simplicial_complex()
print()

# 2. Convert to edge index format
print("Step 2: Converting to edge index format...")
edge_index = simplicial_complex_to_edge_index(sc)
print(f"Edge index shape: {edge_index.shape}")
print(f"Edge index:\n{edge_index}")
print()

# 3. Create random vertex features
print("Step 3: Creating vertex features...")
num_nodes = len(list(sc.skeleton(0)))
in_dim = 4
x = torch.randn(num_nodes, in_dim)
print(f"Vertex features shape: {x.shape}")
print()

# 4. Initialize copresheaf model
print("Step 4: Initializing copresheaf neural network...")
hidden_dim = 8
out_dim = 2
num_layers = 2

model = SimpleCopresheafNN(
    in_dim=in_dim,
    hidden_dim=hidden_dim,
    out_dim=out_dim,
    num_layers=num_layers
)
print(f"Model architecture:")
print(f"  - Input dim: {in_dim}")
print(f"  - Hidden dim: {hidden_dim}")
print(f"  - Output dim: {out_dim}")
print(f"  - Num layers: {num_layers}")
print()

# 5. Forward pass
print("Step 5: Running forward pass...")
edge_index_list = [edge_index]  # Wrap in list for single neighborhood

with torch.no_grad():
    output = model(x, edge_index_list)

print(f"Output shape: {output.shape}")
print(f"Output:\n{output}")
print()

# 6. Training example (optional)
print("Step 6: Simple training example...")

# Create dummy labels
labels = torch.randint(0, out_dim, (num_nodes,))

# Setup optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Train for a few epochs
model.train()
for epoch in range(5):
    optimizer.zero_grad()
    
    # Forward pass
    output = model(x, edge_index_list)
    
    # Compute loss
    loss = criterion(output, labels)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch + 1}/5, Loss: {loss.item():.4f}")

print()
print("=" * 60)
print("Example completed successfully!")
print("=" * 60)